# Crossing_CL analysis

Post-hoc analysis of Crossing_CL models exported by `optimize/nested_optimize_EIANN_Crossing_CL.py`. The task is described in `reinforcement_learning/20261104_crossing_CL_design.md`. In a 4×4 room, a wall has one gap, and three identical goal tiles sit on the far side. Only one tile pays, and the gap's row says which. Tasks A, B and C are trained in blocks, in order.

Everything here works from the greedy episodes recorded during training (`rl_history['val_laps']`), so nothing retrains. The environment is deterministic, so the agent's paths are replayed from the recorded actions.

Export a model with, from the `EIANN` package directory:

```bash
mkdir -p data/Crossing_CL
python -m nested.analyze --framework=serial --config-file-path=<optimize config> \
  --param-file-path=optimize/optimize_params/Crossing_CL/20261104_Crossing_CL_params.yaml --model-key=<key> \
  --output-dir=data/Crossing_CL --label=<key> --num_instances=5 --export --disp
```

Then add the model to `models` below. Figures and videos are saved to `data/rl_plots/Crossing_CL/`, which is gitignored.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

import EIANN.utils as ut
from EIANN.plot import clean_axes, update_plot_defaults
from EIANN.reinforcement_learning.treadmill_metrics import block_rewards, seen_track_rewards
from EIANN.reinforcement_learning.treadmill_plots import SERIES_COLORS, mark_blocks
from EIANN.reinforcement_learning.crossing_metrics import block_outcome_rates, stack_outcomes, cl_summary
from EIANN.reinforcement_learning.crossing_plots import (task_name, plot_matrix, get_outcome_curves,
                                                         plot_crossing_summary, animate_learning,
                                                         animate_greedy_episodes)

%matplotlib inline
update_plot_defaults()

# tasks A, B and C keep SERIES_COLORS[0:3] (as in the block shading), so models use the next colors
MODEL_COLORS = SERIES_COLORS[3:]
plot_dir = '../data/rl_plots/Crossing_CL'
os.makedirs(plot_dir, exist_ok=True)

## Load models

In [ ]:
data_dir = '../data/Crossing_CL'
num_episodes_reward_window = 100  # as in the optimize configs: block metrics average the last episodes of each block

# model name: (network config name, nested label)
models = {'van_bp': ('20261104_EIANN_2_hidden_Crossing_CL_van_bp_relu_SGD_config', 'van_bp_relu_SGD')}


def load_rl_histories(network_name, label):
    """
    The rl_history of every exported seed of a model, by '<seed>_<data_seed>'.
    """
    histories = {}
    for path in sorted(glob.glob(f'{data_dir}/{network_name}/{network_name}_*_{label}.pkl')):
        seed = os.path.basename(path)[len(network_name) + 1:-len(label) - 5]
        histories[seed] = ut.load_network(path, disp=False).rl_history
    return histories


histories = {name: load_rl_histories(*model) for name, model in models.items()}
# models without exported pickles are left out of the plots
missing = [name for name, model_histories in histories.items() if not model_histories]
histories = {name: model_histories for name, model_histories in histories.items() if model_histories}
print('not exported yet:', missing)
{name: list(model_histories) for name, model_histories in histories.items()}

## Block × task matrices

Entry [b, j] is task j at the end of block b, averaged over the validation steps in the last `num_episodes_reward_window` episodes of the block. There are three measures:
- **Success:** the rewarded tile was reached.
- **Reached a tile:** any goal tile was reached (navigation).
- **Choice accuracy:** success ÷ reached, i.e. whether the choice was right given that a tile was reached. Chance is 1/3. The value is undefined (–) when no tile was reached.

The navigation/choice split separates forgetting the route from forgetting which tile pays. Values are the mean over seeds.

In [ ]:
def get_rates(history):
    num_blocks = history['num_tasks'] * history['num_cycles']
    return block_outcome_rates(history['val_laps'], history['val_episodes'], history['episodes_per_task'], num_blocks,
                               num_episodes_reward_window)


rates = {name: {key: np.array([get_rates(history)[key] for history in model_histories.values()])
                for key in ('success', 'navigation', 'choice')}
         for name, model_histories in histories.items()}

for name, model_rates in rates.items():
    num_tasks = next(iter(histories[name].values()))['num_tasks']
    fig, axes = plt.subplots(1, 3, figsize=(6.6, 1.9), layout='constrained')
    for ax, (key, label) in zip(axes, (('success', 'Success'), ('navigation', 'Reached a tile'),
                                        ('choice', 'Choice accuracy'))):
        with np.errstate(invalid='ignore'):
            image = plot_matrix(ax, np.nanmean(model_rates[key], axis=0) if np.isfinite(model_rates[key]).any()
                                else model_rates[key][0], num_tasks)
        ax.set_title(label)
    fig.colorbar(image, ax=axes, shrink=0.8)
    fig.suptitle(f'{name} (n={len(model_rates["success"])} seeds)')
    plt.show()

## Phase rewards and CL summary

`phase{b}_reward` (the optimization objective) is the greedy reward averaged over the tasks seen so far, as with a cumulative validation set in class-incremental MNIST. `phase{b}_success` is the same for success.

ACC, BWT and FWT come from each seed's success matrix (Díaz-Rodríguez et al. 2018):
- **ACC:** mean success over all tasks after the last block.
- **BWT:** mean change on each earlier task between the end of its own block and the end of training. Negative means forgetting.
- **FWT:** success on each task just before its block, minus its success before any training.

Values are the mean ± SEM over seeds.

In [ ]:
def get_reward_matrix(history):
    num_blocks = history['num_tasks'] * history['num_cycles']
    task_rewards = [[lap['episode_return'] for lap in laps] for laps in history['val_laps']]
    return block_rewards(task_rewards, history['val_episodes'], history['episodes_per_task'], num_blocks,
                         num_episodes_reward_window)


phase_values = {name: {'reward': np.array([seen_track_rewards(get_reward_matrix(history))
                                           for history in model_histories.values()]),
                       'success': np.array([seen_track_rewards(matrix) for matrix in rates[name]['success']])}
                for name, model_histories in histories.items()}

fig, axes = plt.subplots(1, 2, figsize=(4.8, 1.8), layout='constrained')
for i, ((name, values), color) in enumerate(zip(phase_values.items(), MODEL_COLORS)):
    for ax, key in zip(axes, ('reward', 'success')):
        phases = np.arange(values[key].shape[1]) + 0.1 * (i - (len(phase_values) - 1) / 2)
        ax.errorbar(phases, values[key].mean(axis=0), yerr=values[key].std(axis=0) / np.sqrt(len(values[key])),
                    color=color, marker='o', markersize=3, linewidth=1, capsize=2,
                    label=f'{name} (n={len(values[key])})')
for ax, label in zip(axes, ('Reward, seen tasks', 'Success, seen tasks')):
    ax.set_xticks(range(values['reward'].shape[1]),
                  [f'{block} ({task_name(block)})' for block in range(values['reward'].shape[1])])
    ax.set_xlabel('Block (trained on)')
    ax.set_ylabel(label)
axes[0].legend()
clean_axes(axes)
plt.show()


def get_cl_summary(history, success_matrix):
    # baseline for FWT: greedy success of every task at the first validation step, before (almost) any training
    baseline = (stack_outcomes(history['val_laps'][:1]) == 'correct')[0].astype(float)
    return cl_summary(success_matrix, baseline)


summary = []
for name, model_histories in histories.items():
    seeds = pd.DataFrame([get_cl_summary(history, matrix)
                          for history, matrix in zip(model_histories.values(), rates[name]['success'])])
    summary.append({'model': name, 'n': len(seeds),
                    **{f'{metric}': f'{seeds[metric].mean():.2f} ± {seeds[metric].sem():.2f}'
                       for metric in seeds.columns}})
pd.DataFrame(summary).set_index('model')

## Over training

Each measure on every task over training: the mean over seeds of a running mean over `window` validation steps, ± SEM. Blocks are shaded by the task they train on.

In [ ]:
window = 10  # validation steps


def plot_mean_sem(ax, episodes, values, color, label=None):
    with np.errstate(invalid='ignore'):
        mean = np.nanmean(values, axis=0)
        sem = np.nanstd(values, axis=0) / np.sqrt(np.sum(np.isfinite(values), axis=0))
    ax.plot(episodes, mean, color=color, linewidth=1, label=label)
    ax.fill_between(episodes, mean - sem, mean + sem, color=color, alpha=0.2, linewidth=0)


for name, model_histories in histories.items():
    first = next(iter(model_histories.values()))
    curves = [get_outcome_curves(history, window) for history in model_histories.values()]
    episodes = curves[0][0]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 1.8), sharey=True, layout='constrained')
    for ax, (key, label) in zip(axes, (('success', 'Success'), ('navigation', 'Reached a tile'),
                                        ('choice', 'Choice accuracy'))):
        mark_blocks(ax, episodes, first['episodes_per_task'], first['num_tasks'])
        values = np.array([seed_curves[key] for _, seed_curves in curves])
        for task in range(first['num_tasks']):
            plot_mean_sem(ax, episodes, values[:, :, task], SERIES_COLORS[task], label=f'Task {task_name(task)}')
        ax.set_title(label)
        ax.set_xlabel('Episode')
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].set_ylabel('Fraction of greedy episodes')
    axes[-1].legend(frameon=False, loc='upper left', bbox_to_anchor=(1, 1))
    fig.suptitle(f'{name} (n={len(curves)} seeds)')
    clean_axes(axes)
    plt.show()

## One seed

The standard plots of one seed. `nested.analyze --plot` makes the same plots.

In [ ]:
model, seed = 'van_bp', list(histories['van_bp'])[0]
history = histories[model][seed]
plot_crossing_summary(history, num_episodes_reward_window, title=f'{model} {seed}')

## Videos

**Learning over training** (`animate_learning`): one frame per validation step (every 10 episodes in the optimize configs).
- Each room shows that step's greedy path on its task. The path darkens towards its end, and the end is marked by outcome: green correct, red wrong, gray timeout.
- The star is the rewarded tile.
- The task being trained is framed in its color.
- Below, the success curves, with a line at the current episode.

The video is saved as a GIF; pass a `.mp4` path instead if ffmpeg is installed.

In [ ]:
learning_path = f'{plot_dir}/{model}_{seed}_learning.gif'
animate_learning(history, window=window, title=f'{model} {seed}', save_path=learning_path)
plt.close('all')
display(Image(filename=learning_path))

**Greedy episodes, step by step** (`animate_greedy_episodes`), in MiniGrid's own rendering:
- The red triangle is the agent, pointing the way it last moved.
- Green tiles are the goal tiles, and the star is the one that pays.
- The white square is the 3×3 neighborhood the agent observes.

One video is made at the end of every block, which shows what the network does on all three tasks after training on each.

In [ ]:
episodes = np.asarray(history['val_episodes'])
num_blocks = history['num_tasks'] * history['num_cycles']
for block in range(num_blocks):
    # last validation step of the block
    val_step = int(np.searchsorted(episodes, (block + 1) * history['episodes_per_task'] - 1, side='right') - 1)
    path = f'{plot_dir}/{model}_{seed}_greedy_block{block}.gif'
    animate_greedy_episodes(history, val_step=val_step,
                            title=f'{model} {seed}, end of block {block} (task {task_name(block % history["num_tasks"])})',
                            save_path=path)
    plt.close('all')
    display(Image(filename=path))